<a href="https://colab.research.google.com/github/LBJLincoln/nomos-nba-agent/blob/main/colab/tabicl_oracle_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Oracle — clean Colab T4 pipeline

Pulls 9 seasons (11,532 games) + 7 enriched data files from HF, runs `engine.build()` with all feeds wired (referee/player_merged/quarter/market/odds/tracking), trains XGBoost+LightGBM on ALL alive features and TabICL on top-186, picks winner by **walk-forward held-out Brier** (last 15% as time-ordered holdout — overfit-detector), archives to HF, promotes if Brier < 0.22087.

Setup: Runtime→GPU T4. Tools→Secrets→`HF_TOKEN`. Then Run All. ~30-45 min.

In [ ]:
# Cell 1: install + token + GPU assert
!pip install -q tabicl xgboost lightgbm huggingface_hub scikit-learn==1.6.1 2>&1 | tail -3
import os, sys, json, time, pickle, warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timezone
from pathlib import Path
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print('token len=', len(HF_TOKEN))

# 2026-04-28 — hard-fail if GPU not attached so the run never silently spends
# 30+ min on CPU. TabICL defaults to CPU when device= is omitted; we now pass
# DEVICE explicitly to every TabICLClassifier(...) below.
import torch, subprocess
assert torch.cuda.is_available(), \
    'No CUDA. Runtime -> Change runtime type -> T4 GPU -> Save -> Connect -> re-run.'
DEVICE = 'cuda'
print(f'GPU: {torch.cuda.get_device_name(0)} '
      f'mem={torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
subprocess.run(['nvidia-smi', '-L'], check=False)

In [2]:
# Cell 2: pull ALL files (9 seasons + 7 data files)
from huggingface_hub import hf_hub_download
DATASET = 'LBJLincoln26/nba-feature-cache'
DATA_DIR = Path('/content/feature_data')
DATA_DIR.mkdir(exist_ok=True)

DATA_FILES = ['engine.py','referee_data.json','player_data_merged.json',
              'quarter_data.json','polymarket_data.json','full-odds-2025-26.json',
              'tracking_data.json']
SEASON_FILES = [f'games-{y}.json' for y in
                ['2017-18','2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25','2025-26']]
for f in DATA_FILES + SEASON_FILES:
    p = hf_hub_download(repo_id=DATASET, filename=f'feature_data/{f}', repo_type='dataset', token=HF_TOKEN)
    target = DATA_DIR / f
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(p)
    print(f'  {f:35s} {Path(p).stat().st_size/1024/1024:6.2f} MB')
print(f'\n{len(DATA_FILES)} data + {len(SEASON_FILES)} season files ready')

engine.py: 0.00B [00:00, ?B/s]

  engine.py                             0.48 MB


referee_data.json: 0.00B [00:00, ?B/s]

  referee_data.json                     0.31 MB


feature_data/player_data_merged.json:   0%|          | 0.00/13.9M [00:00<?, ?B/s]

...

  games-2023-24.json                    0.53 MB


games-2024-25.json: 0.00B [00:00, ?B/s]

  games-2024-25.json                    1.24 MB


games-2025-26.json: 0.00B [00:00, ?B/s]

  games-2025-26.json                    1.11 MB

7 data + 9 season files ready


In [3]:
# Cell 3: load engine + all data feeds, concat all seasons
import importlib.util
spec = importlib.util.spec_from_file_location('engine', str(DATA_DIR / 'engine.py'))
engine_mod = importlib.util.module_from_spec(spec)
sys.modules['engine'] = engine_mod
spec.loader.exec_module(engine_mod)
print('engine_version:', engine_mod.ENGINE_VERSION)

# 1. Concat all 9 seasons
games = []
for season_file in SEASON_FILES:
    raw = json.loads((DATA_DIR / season_file).read_text())
    season_games = raw.get('games', raw) if isinstance(raw, dict) else raw
    games.extend(season_games)
    print(f'  {season_file}: {len(season_games)} games')
print(f'  TOTAL: {len(games)} games')

# 2. Per-game referee crew rolling stats (leakage-safe)
referee_data = json.loads((DATA_DIR / 'referee_data.json').read_text())
tracking_data = json.loads((DATA_DIR / 'tracking_data.json').read_text())

# 3. Per-(team,date) merged player+coach+altitude+tracking+synergy+position+streaks (234 fields)
raw_pd = json.loads((DATA_DIR / 'player_data_merged.json').read_text())
player_data = {}
for k, v in raw_pd.items():
    if '|' in k:
        team, date = k.split('|', 1)
        player_data[(team, date)] = v
    else:
        player_data[k] = v

raw_qd = json.loads((DATA_DIR / 'quarter_data.json').read_text())
quarter_data = {tuple(k.split('|',1)): v for k, v in raw_qd.items()}

# 4. Polymarket per game_id
raw_pm = json.loads((DATA_DIR / 'polymarket_data.json').read_text())
market_data = {}
for g in games:
    gid = g.get('game_id', '')
    d = (g.get('game_date') or '')[:10]
    h = (g.get('home',{}) or {}).get('team_abbr','') if isinstance(g.get('home'), dict) else ''
    a = (g.get('away',{}) or {}).get('team_abbr','') if isinstance(g.get('away'), dict) else ''
    key = f'{d}|{h}|{a}'
    if key in raw_pm:
        market_data[gid] = raw_pm[key]

# 5. Full-odds keys are 'YYYY-MM-DD_AWAY@HOME', flatten 'base' subdict
raw_odds = json.loads((DATA_DIR / 'full-odds-2025-26.json').read_text())
odds_data = {}
for k, v in raw_odds.items():
    if not isinstance(v, dict) or '_' not in k or '@' not in k:
        continue
    try:
        date_part, matchup = k.split('_', 1)
        away, home = matchup.split('@', 1)
    except ValueError:
        continue
    base = v.get('base', {})
    odds_data[(date_part[:10], home, away)] = dict(base)

print(f'\nfeeds: ref={len(referee_data)} player={len(player_data)} quarter={len(quarter_data)} market={len(market_data)} odds={len(odds_data)} tracking={len(tracking_data)}')
assert len(referee_data) > 100, 'referee_data empty'
assert len(player_data) > 100, 'player_data empty'
assert len(odds_data) > 100, 'odds_data empty (parser bug?)'

engine_version: v3.2-67cat
  games-2017-18.json: 1312 games
  games-2018-19.json: 1312 games
  games-2019-20.json: 1142 games
  games-2020-21.json: 1165 games
  games-2021-22.json: 1317 games
  games-2022-23.json: 1314 games
  games-2023-24.json: 1312 games
  games-2024-25.json: 1401 games
  games-2025-26.json: 1257 games
  TOTAL: 11532 games

feeds: ref=1181 player=2395 quarter=2343 market=1186 odds=802 tracking=30


In [ ]:
# Cell 4: build feature matrix on ALL 11,500 games × 7,246 cols
import numpy as np
engine = engine_mod.NBAFeatureEngine()
t0 = time.time()
X, y, feature_names = engine.build(
    games,
    referee_data=referee_data, player_data=player_data,
    quarter_data=quarter_data, market_data=market_data,
    odds_data=odds_data, tracking_data=tracking_data,
)
elapsed = time.time() - t0
X = np.nan_to_num(np.array(X, dtype=np.float32))
y = np.array(y, dtype=np.float64)
print(f'build: {X.shape[0]} games × {X.shape[1]} cols in {elapsed:.0f}s; y_mean={y.mean():.3f}')

variances = X.var(axis=0)
alive_mask = variances > 1e-10
alive_idx = np.where(alive_mask)[0]
print(f'alive: {len(alive_idx)}/{X.shape[1]} ({len(alive_idx)/X.shape[1]*100:.1f}%)')

X_full = X[:, alive_idx].astype(np.float32)  # PURE numpy — no DataFrame, no warning
ranked = sorted(alive_idx, key=lambda i: -variances[i])
top186_idx = ranked[:186]
X_186 = X[:, top186_idx].astype(np.float32)
print(f'X_full (XGB/LGBM): {X_full.shape}    X_186 (TabICL): {X_186.shape}')
assert X.shape[0] > 1000, f'too few games — only {X.shape[0]} survived engine filters'

In [ ]:
# Cell 5: walk-forward holdout + 10-fold CPCV setup
# Walk-forward holdout = last 15% of time-ordered games. Tests overfit honestly.
from sklearn.metrics import brier_score_loss
RANDOM_STATE = 1337
N = len(X_full)
HOLDOUT_FRAC = 0.15
HOLDOUT_START = int(N * (1 - HOLDOUT_FRAC))
X_train_full = X_full[:HOLDOUT_START]
y_train = y[:HOLDOUT_START]
X_holdout_full = X_full[HOLDOUT_START:]
y_holdout = y[HOLDOUT_START:]
X_train_186 = X_186[:HOLDOUT_START]
X_holdout_186 = X_186[HOLDOUT_START:]
print(f'TRAIN: {X_train_full.shape}    HOLDOUT (last {HOLDOUT_FRAC*100:.0f}%): {X_holdout_full.shape}')
print(f'  train y_mean={y_train.mean():.3f}  holdout y_mean={y_holdout.mean():.3f}')

# CPCV on TRAIN only
N_FOLDS = 10
EMBARGO = max(1, int(len(X_train_full) * 0.02))
FOLD_SIZE = len(X_train_full) // N_FOLDS
def cpcv_folds(n):
    for k in range(N_FOLDS):
        lo = k * FOLD_SIZE
        hi = lo + FOLD_SIZE if k < N_FOLDS - 1 else n
        m = np.ones(n, dtype=bool)
        m[max(0, lo - EMBARGO):min(n, hi + EMBARGO)] = False
        yield np.where(m)[0], np.arange(lo, hi)

all_results = []  # each: {model, brier_cv, brier_holdout, ...}

In [ ]:
# Cell 6: XGBoost — CV brier + walk-forward holdout brier
import xgboost as xgb
fold_briers = []
oof = np.zeros(len(X_train_full))
t0 = time.time()
for tr, te in cpcv_folds(len(X_train_full)):
    m = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.6,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, tree_method='hist', device='cuda',
        verbosity=0,
    )
    m.fit(X_train_full[tr], y_train[tr])
    p = m.predict_proba(X_train_full[te])[:, 1]
    oof[te] = p
    fold_briers.append(float(brier_score_loss(y_train[te], p)))
xgb_cv = float(np.mean(fold_briers))
elapsed_cv = time.time() - t0

# Walk-forward holdout test (train on ALL train rows, predict held-out future)
m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
                     random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)
m.fit(X_train_full, y_train)
p_holdout = m.predict_proba(X_holdout_full)[:, 1]
xgb_holdout = float(brier_score_loss(y_holdout, p_holdout))

print(f'XGBoost: CV={xgb_cv:.5f} | walk-forward holdout={xgb_holdout:.5f} | gap={xgb_holdout-xgb_cv:+.5f}')
if xgb_holdout - xgb_cv > 0.020:
    print(f'  ⚠️  CV >20bp better than holdout → CV may be overfit')
all_results.append({'model':'xgboost','features':X_full.shape[1],
                    'brier_cv':xgb_cv,'brier_holdout':xgb_holdout,
                    'fold_briers':fold_briers,'oof':oof.tolist(),
                    'holdout_preds':p_holdout.tolist()})

In [ ]:
# Cell 7: LightGBM — same protocol
import lightgbm as lgb
fold_briers = []
oof = np.zeros(len(X_train_full))
t0 = time.time()
for tr, te in cpcv_folds(len(X_train_full)):
    m = lgb.LGBMClassifier(
        n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.6,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, verbose=-1, force_col_wise=True,
    )
    m.fit(X_train_full[tr], y_train[tr])
    p = m.predict_proba(X_train_full[te])[:, 1]
    oof[te] = p
    fold_briers.append(float(brier_score_loss(y_train[te], p)))
lgb_cv = float(np.mean(fold_briers))

m = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
                       random_state=RANDOM_STATE, verbose=-1, force_col_wise=True)
m.fit(X_train_full, y_train)
p_holdout = m.predict_proba(X_holdout_full)[:, 1]
lgb_holdout = float(brier_score_loss(y_holdout, p_holdout))

print(f'LightGBM: CV={lgb_cv:.5f} | walk-forward holdout={lgb_holdout:.5f} | gap={lgb_holdout-lgb_cv:+.5f}')
if lgb_holdout - lgb_cv > 0.020:
    print(f'  ⚠️  CV >20bp better than holdout → likely overfit')
all_results.append({'model':'lightgbm','features':X_full.shape[1],
                    'brier_cv':lgb_cv,'brier_holdout':lgb_holdout,
                    'fold_briers':fold_briers,'oof':oof.tolist(),
                    'holdout_preds':p_holdout.tolist()})

In [ ]:
# Cell 8: TabICL — top-186 only (architectural ctx limit). Same CV+holdout protocol.
# 2026-04-28 — device=DEVICE added; without it TabICL silently runs on CPU even
# with T4 attached (cost ~1h on prior runs).
from tabicl import TabICLClassifier
best_tabicl = None
for ctx, temp in [(2048, 1.0), (3072, 1.0), (2048, 1.08)]:
    fold_briers = []
    oof = np.zeros(len(X_train_186))
    t0 = time.time()
    for tr, te in cpcv_folds(len(X_train_186)):
        m = TabICLClassifier(n_estimators=1, softmax_temperature=temp,
                             device=DEVICE, random_state=RANDOM_STATE)
        sub_tr = tr[-ctx:]
        m.fit(X_train_186[sub_tr], y_train[sub_tr])
        p = m.predict_proba(X_train_186[te])[:, 1]
        oof[te] = p
        fold_briers.append(float(brier_score_loss(y_train[te], p)))
    cv = float(np.mean(fold_briers))
    print(f'TabICL ctx={ctx} temp={temp}: CV={cv:.5f} in {time.time()-t0:.0f}s')
    if best_tabicl is None or cv < best_tabicl['brier_cv']:
        best_tabicl = {'model':'tabicl','features':186,'brier_cv':cv,'ctx':ctx,'temp':temp,
                       'fold_briers':fold_briers,'oof':oof.tolist()}

# Holdout for best TabICL
m = TabICLClassifier(n_estimators=1, softmax_temperature=best_tabicl['temp'],
                     device=DEVICE, random_state=RANDOM_STATE)
m.fit(X_train_186[-best_tabicl['ctx']:], y_train[-best_tabicl['ctx']:])
p_holdout = m.predict_proba(X_holdout_186)[:, 1]
best_tabicl['brier_holdout'] = float(brier_score_loss(y_holdout, p_holdout))
best_tabicl['holdout_preds'] = p_holdout.tolist()
print(f"best TabICL: ctx={best_tabicl['ctx']} temp={best_tabicl['temp']} CV={best_tabicl['brier_cv']:.5f} holdout={best_tabicl['brier_holdout']:.5f}")
all_results.append(best_tabicl)

In [ ]:
# Cell 9: pick winner BY HOLDOUT BRIER (overfit-safe), isotonic-calibrate, retrain on ALL
from sklearn.isotonic import IsotonicRegression

# Winner = lowest walk-forward holdout brier (the honest metric)
winner = min(all_results, key=lambda r: r['brier_holdout'])
print(f"\n{'='*60}")
print(f"WINNER (by walk-forward holdout): {winner['model']} on {winner['features']} features")
print(f"  CV brier:      {winner['brier_cv']:.5f}")
print(f"  Holdout brier: {winner['brier_holdout']:.5f}")
print(f"{'='*60}")
for r in all_results:
    gap = r['brier_holdout'] - r['brier_cv']
    flag = '⚠️ ' if gap > 0.020 else '   '
    print(f"  {r['model']:10s} {r['features']:>5d} feats | CV {r['brier_cv']:.5f} | holdout {r['brier_holdout']:.5f} | gap {gap:+.5f} {flag}")

oof = np.array(winner['oof'])
iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y_train)
raw_b = brier_score_loss(y_train, oof)
cal_b = brier_score_loss(y_train, iso.predict(oof))
print(f'\nisotonic: raw={raw_b:.5f} → calibrated={cal_b:.5f}')

# Retrain WINNER on ALL data (train + holdout) for production
if winner['model'] == 'xgboost':
    final = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
                              random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)
    final.fit(X_full, y)
    final_features = [feature_names[i] for i in alive_idx]
    final_indices = list(map(int, alive_idx))
elif winner['model'] == 'lightgbm':
    final = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,
                               subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
                               random_state=RANDOM_STATE, verbose=-1, force_col_wise=True)
    final.fit(X_full, y)
    final_features = [feature_names[i] for i in alive_idx]
    final_indices = list(map(int, alive_idx))
else:
    # 2026-04-28 — device=DEVICE added (was missing → silent CPU fallback)
    final = TabICLClassifier(n_estimators=1, softmax_temperature=winner['temp'],
                             device=DEVICE, random_state=RANDOM_STATE)
    final.fit(X_186[-winner['ctx']:], y[-winner['ctx']:])
    final_features = [feature_names[i] for i in top186_idx]
    final_indices = list(map(int, top186_idx))
print(f'\nfinal {winner["model"]} retrained on all {len(y)} games × {len(final_features)} features')

In [ ]:
# Cell 10: archive + auto-promote (uses holdout brier, not CV) + hot-reload oracle Space
from huggingface_hub import HfApi, hf_hub_download
utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
bundle = {
    'model': final, 'calibrator': iso,
    'feature_indices': final_indices, 'feature_names': final_features,
    'cv_brier_mean': winner['brier_cv'],
    'cv_brier_per_fold': winner.get('fold_briers', []),
    'holdout_brier': winner['brier_holdout'],
    'config': {
        'model_type': winner['model'], 'n_features': len(final_features),
        'random_state': RANDOM_STATE, 'cpcv_folds': N_FOLDS,
        'engine_version': engine_mod.ENGINE_VERSION,
        'features_alive_total': int(len(alive_idx)),
        'features_engine_total': int(X.shape[1]),
        'all_results_summary': [{'model': r['model'], 'features': r['features'],
                                  'brier_cv': r['brier_cv'], 'brier_holdout': r['brier_holdout']}
                                 for r in all_results],
        'walk_forward_holdout_frac': HOLDOUT_FRAC,
        'holdout_n_games': len(y_holdout),
    },
    'n_samples': int(X.shape[0]),
    'trained_at': utc, 'trained_on': f'colab-multi-{winner["model"]}',
}
if winner['model'] == 'tabicl':
    bundle['config']['ctx_size'] = winner['ctx']
    bundle['config']['softmax_temperature'] = winner['temp']
PKL = f'/tmp/oracle-{winner["model"]}-{utc}.pkl'
with open(PKL, 'wb') as f:
    pickle.dump(bundle, f)
print(f'pkl: {PKL} ({Path(PKL).stat().st_size/1024/1024:.1f} MB)')

api = HfApi(token=HF_TOKEN)
api.create_repo('LBJLincoln26/nba-oracle-archive', repo_type='dataset', private=False, exist_ok=True)
api.upload_file(
    path_or_fileobj=PKL,
    path_in_repo=f'colab-multi-{winner["model"]}-{utc}.pkl',
    repo_id='LBJLincoln26/nba-oracle-archive', repo_type='dataset',
    commit_message=f'[colab-multi] {winner["model"]} CV {winner["brier_cv"]:.5f} holdout {winner["brier_holdout"]:.5f} alive={len(alive_idx)}/{X.shape[1]}',
)
print('archived')

try:
    cur = json.load(open(hf_hub_download(
        repo_id='LBJLincoln26/nba-oracle-model', filename='summary.json',
        repo_type='dataset', token=HF_TOKEN)))
    cur_brier = float(cur.get('cv_brier_mean', 0.99))
except Exception:
    cur_brier = 0.99
print(f'production: {cur_brier:.5f}')

# Promote based on HOLDOUT brier — not CV (overfit-safe)
if winner['brier_holdout'] < cur_brier:
    api.upload_file(
        path_or_fileobj=PKL, path_in_repo='nba-oracle.pkl',
        repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset',
        commit_message=f'[PROMOTE] {winner["model"]} holdout {winner["brier_holdout"]:.5f} (was {cur_brier:.5f}, CV {winner["brier_cv"]:.5f})',
    )
    summary = {k: v for k, v in bundle.items() if k not in ('model', 'calibrator')}
    summary['promoted_from_brier'] = cur_brier
    summary['cv_brier_mean'] = winner['brier_holdout']  # store HOLDOUT as production brier
    api.upload_file(
        path_or_fileobj=json.dumps(summary, indent=2, default=str).encode(),
        path_in_repo='summary.json', repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset',
        commit_message='[PROMOTE colab-multi] summary update',
    )
    print(f'\n*** PROMOTED: {winner["model"]} holdout {winner["brier_holdout"]:.5f} (was {cur_brier:.5f}) ***')

    # 2026-04-28 — kick the live oracle Space to hot-reload the new bundle
    # (otherwise it serves the old model until next restart).
    import requests
    try:
        r = requests.post('https://lbjlincoln26-nba-oracle.hf.space/api/reload', timeout=30)
        print(f'oracle hot-reload: {r.status_code} {r.json()}')
    except Exception as e:
        print(f'oracle reload failed (non-fatal — 5min etag watcher will pick it up): {e}')
else:
    print(f'\nNot promoted: holdout {winner["brier_holdout"]:.5f} >= production {cur_brier:.5f}')